# Mini Project

Solutions from:   

Name: **Manoj Billur Nagendra Prasad**   
Matriculation Number: 2886261

Name: **Steffi Stephen**  
Matriculation Number: 4109600

Goal : The objective of this notebook is to convert the RDF knowledge graph into a tabular dataset suitable for machine learning.

Instead of directly converting every RDF predicate into a feature, we first analyze the neighborhood of the labeled Person nodes, identify informative predicates, justify feature engineering decisions, and then construct the final feature matrix.

1. Understanding Person Nodes

2. Person Neighborhood Analysis

4. Candidate Features

5. Feature Design Decisions (means feautures to numbers)

6. RDF → DataFrame

7. Final Feature Matrix


This notebook uses the following inputs:

| File | Description |
|------|-------------|
| `aifbfixed_complete.n3` | RDF knowledge graph |
| `trainingSet.tsv` | Training dataset |
| `testSet.tsv` | Test dataset |
| `completeDataset.tsv` | Complete labelled dataset |

This notebook generates the following files:

| File | Description |
|------|-------------|
| `feature_matrix.csv` | Final feature matrix for machine learning |
| `feature_matrix.pkl` | Serialized feature matrix |
| `feature_summary.csv` | Summary of engineered features |
| `outgoing_predicate_coverage.csv` | Coverage of outgoing predicates |
| `incoming_predicate_coverage.csv` | Coverage of incoming predicates |

In [1]:
# =============================================================================
# Project Setup
# =============================================================================

from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

# Import project paths
from utils.paths import *

# Import helper functions
from utils.helper_functions import (
    get_entity_name,
    get_predicate_name,
    print_triple,
    has_homepage,
    get_publication_count,
    get_works_at_project_count,
    get_number_of_projects,
    clean_column_name
)

In [2]:
# =============================================================================
# Standard Library
# =============================================================================

from collections import Counter, defaultdict
import itertools

# =============================================================================
# Third-Party Libraries
# =============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import re

from rdflib import Graph
from IPython.display import display, HTML

In [3]:
graph = Graph()

graph.parse(RDF_FILE, format="n3")

print("RDF graph loaded successfully.")

RDF graph loaded successfully.


In [4]:
print("First 10 Human-Readable RDF Triples\n")

for i, (subject, predicate, obj) in enumerate(graph):

    print(f"Triple {i+1}")

    print_triple(subject, predicate, obj)

    if i == 5:
        break

First 10 Human-Readable RDF Triples

Triple 1
Subject   : Publikationen(id2instance)
Predicate : pages
Object    : 662-672
--------------------------------------------------------------------------------
Triple 2
Subject   : Publikationen(id92instance)
Predicate : publication
Object    : Publikationen(id114instance)
--------------------------------------------------------------------------------
Triple 3
Subject   : Projekte(id18instance)
Predicate : name
Object    : PEdit - Programmierumgebung fÃ¼r parallele Systeme
--------------------------------------------------------------------------------
Triple 4
Subject   : Personen(id2096instance)
Predicate : publication
Object    : Publikationen(id1134instance)
--------------------------------------------------------------------------------
Triple 5
Subject   : Publikationen(id388instance)
Predicate : type
Object    : Person
--------------------------------------------------------------------------------
Triple 6
Subject   : Publikationen(i

In [5]:
train_df = pd.read_csv(TRAIN_FILE, sep="\t")

In [6]:
train_df.head()

,person,id,label_affiliation
0,http://www.aifb.uni-karlsruhe.de/Personen/view...,2.0,http://www.aifb.uni-karlsruhe.de/Forschungsgru...
1,http://www.aifb.uni-karlsruhe.de/Personen/view...,3.0,http://www.aifb.uni-karlsruhe.de/Forschungsgru...
2,http://www.aifb.uni-karlsruhe.de/Personen/view...,4.0,http://www.aifb.uni-karlsruhe.de/Forschungsgru...
3,http://www.aifb.uni-karlsruhe.de/Personen/view...,5.0,http://www.aifb.uni-karlsruhe.de/Forschungsgru...
4,http://www.aifb.uni-karlsruhe.de/Personen/view...,6.0,http://www.aifb.uni-karlsruhe.de/Forschungsgru...


In [7]:
complete_df = pd.read_csv(COMPLETE_DATASET_FILE, sep="\t")

In [8]:
complete_df.head()

,id,person,label_affiliation
0,1,http://www.aifb.uni-karlsruhe.de/Personen/view...,http://www.aifb.uni-karlsruhe.de/Forschungsgru...
1,2,http://www.aifb.uni-karlsruhe.de/Personen/view...,http://www.aifb.uni-karlsruhe.de/Forschungsgru...
2,3,http://www.aifb.uni-karlsruhe.de/Personen/view...,http://www.aifb.uni-karlsruhe.de/Forschungsgru...
3,4,http://www.aifb.uni-karlsruhe.de/Personen/view...,http://www.aifb.uni-karlsruhe.de/Forschungsgru...
4,5,http://www.aifb.uni-karlsruhe.de/Personen/view...,http://www.aifb.uni-karlsruhe.de/Forschungsgru...


# 1) Understanding Person Nodes

## Example 1 (Individual Exploration)

In [9]:
# Select the first labeled person from the training dataset
example_person = train_df.iloc[0]["person"]

print("Example Person:\n")
print(example_person)

Example Person:

http://www.aifb.uni-karlsruhe.de/Personen/viewPersonOWL/id1909instance


In [10]:
print("="*80)
print("Outgoing Triples")
print("="*80)

for subject, predicate, obj in graph:

    if str(subject) == example_person:
        print_triple(subject, predicate, obj)

Outgoing Triples
Subject   : Personen(id1909instance)
Predicate : type
Object    : Person
--------------------------------------------------------------------------------
Subject   : Personen(id1909instance)
Predicate : name
Object    : Shenqing Yang
--------------------------------------------------------------------------------
Subject   : Personen(id1909instance)
Predicate : affiliation
Object    : Forschungsgruppen(id1instance)
--------------------------------------------------------------------------------
Subject   : Personen(id1909instance)
Predicate : fax
Object    : 
--------------------------------------------------------------------------------
Subject   : Personen(id1909instance)
Predicate : phone
Object    : 
--------------------------------------------------------------------------------


In [11]:
print("="*80)
print("Incoming Triples")
print("="*80)

for subject, predicate, obj in graph:

    if str(obj) == example_person:
        print_triple(subject, predicate, obj)

Incoming Triples


In [12]:
# checking whether this person appears elsewhere in the graph.
count = 0

for s, p, o in graph:

    if str(s) == example_person or str(o) == example_person:
        count += 1

print("Total triples involving this person:", count)

Total triples involving this person: 5


## Example 2 (Individual Exploration)

In [13]:
# Select another labeled person
example_person = train_df.iloc[1]["person"]

print("Example Person:\n")
print(example_person)

Example Person:

http://www.aifb.uni-karlsruhe.de/Personen/viewPersonOWL/id2040instance


In [14]:
print("=" * 80)
print("Outgoing Triples")
print("=" * 80)

for subject, predicate, obj in graph:

    if str(subject) == example_person:
        print_triple(subject, predicate, obj)

Outgoing Triples
Subject   : Personen(id2040instance)
Predicate : affiliation
Object    : Forschungsgruppen(id3instance)
--------------------------------------------------------------------------------
Subject   : Personen(id2040instance)
Predicate : publication
Object    : Publikationen(id1034instance)
--------------------------------------------------------------------------------
Subject   : Personen(id2040instance)
Predicate : fax
Object    : +49 (721) 693717
--------------------------------------------------------------------------------
Subject   : Personen(id2040instance)
Predicate : publication
Object    : Publikationen(id746instance)
--------------------------------------------------------------------------------
Subject   : Personen(id2040instance)
Predicate : publication
Object    : Publikationen(id1313instance)
--------------------------------------------------------------------------------
Subject   : Personen(id2040instance)
Predicate : phone
Object    : +49 (721) 608 894

In [15]:
print("=" * 80)
print("Incoming Triples")
print("=" * 80)

for subject, predicate, obj in graph:

    if str(obj) == example_person:
        print_triple(subject, predicate, obj)

Incoming Triples
Subject   : Publikationen(id1034instance)
Predicate : author
Object    : Personen(id2040instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id746instance)
Predicate : author
Object    : Personen(id2040instance)
--------------------------------------------------------------------------------
Subject   : Forschungsgebiete(id19instance)
Predicate : isWorkedOnBy
Object    : Personen(id2040instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id1313instance)
Predicate : author
Object    : Personen(id2040instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id864instance)
Predicate : author
Object    : Personen(id2040instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id967instance)
Predicate : author
Object    : Personen(i

In [16]:
count = 0

for s, p, o in graph:
    if str(s) == example_person or str(o) == example_person:
        count += 1

print("Total triples involving this person:", count)

Total triples involving this person: 24


## Example 3 (Individual Exploration)---important

In [17]:
example_person = "http://www.aifb.uni-karlsruhe.de/Personen/viewPersonOWL/id98instance"

print(example_person)

http://www.aifb.uni-karlsruhe.de/Personen/viewPersonOWL/id98instance


In [18]:
print("=" * 80)
print("Outgoing Triples")
print("=" * 80)

for subject, predicate, obj in graph:

    if str(subject) == example_person:
        print_triple(subject, predicate, obj)

Outgoing Triples
Subject   : Personen(id98instance)
Predicate : publication
Object    : Publikationen(id869instance)
--------------------------------------------------------------------------------
Subject   : Personen(id98instance)
Predicate : publication
Object    : Publikationen(id1004instance)
--------------------------------------------------------------------------------
Subject   : Personen(id98instance)
Predicate : publication
Object    : Publikationen(id952instance)
--------------------------------------------------------------------------------
Subject   : Personen(id98instance)
Predicate : publication
Object    : Publikationen(id977instance)
--------------------------------------------------------------------------------
Subject   : Personen(id98instance)
Predicate : publication
Object    : Publikationen(id922instance)
--------------------------------------------------------------------------------
Subject   : Personen(id98instance)
Predicate : phone
Object    : +49 (721) 60

In [19]:
print("=" * 80)
print("Incoming Triples")
print("=" * 80)

for subject, predicate, obj in graph:

    if str(obj) == example_person:
        print_triple(subject, predicate, obj)

Incoming Triples
Subject   : Publikationen(id907instance)
Predicate : author
Object    : Personen(id98instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id430instance)
Predicate : author
Object    : Personen(id98instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id1244instance)
Predicate : author
Object    : Personen(id98instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id146instance)
Predicate : author
Object    : Personen(id98instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id876instance)
Predicate : author
Object    : Personen(id98instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id1282instance)
Predicate : author
Object    : Personen(id98instance)
------

In [20]:
count = 0

for s, p, o in graph:

    if str(s) == example_person or str(o) == example_person:
        count += 1

print("Total triples involving this person:", count)

Total triples involving this person: 155


## Complete Dataset Exploration

To answer the question: What information do the 176 labeled persons generally have, (I mean this is with related to the Outgoing Triples)

Eg: Personen(id2040instance) ----publication----> Publikationen(id746instance)

In [21]:
# All labeled persons
labeled_persons = set(complete_df["person"])

# Stores which persons have which outgoing predicate
predicate_persons = defaultdict(set)

for subject, predicate, obj in graph:

    if str(subject) in labeled_persons:

        predicate_name = get_predicate_name(predicate)

        predicate_persons[predicate_name].add(str(subject))

In [22]:
predicate_summary = []

for predicate, persons in predicate_persons.items():

    predicate_summary.append({
        "Outgoing Predicate": predicate,
        "Persons": len(persons),
        "Coverage (%)": round(len(persons) / len(labeled_persons) * 100, 2)
    })

predicate_df = (
    pd.DataFrame(predicate_summary)
      .sort_values("Persons", ascending=False)
      .reset_index(drop=True)
)

predicate_df

,Outgoing Predicate,Persons,Coverage (%)
0,affiliation,176,100.00
1,name,176,100.00
2,type,176,100.00
3,phone,176,100.00
4,fax,176,100.00
5,publication,124,70.45
6,photo,113,64.20
7,worksAtProject,77,43.75
8,homepage,72,40.91


To answer the question: What information do other entities store about the Person?, (I mean this is with related to the Incoming Triples)

Eg: Publikationen(id864instance) ----author----> Personen(id2040instance)

In [23]:
# Stores which persons have which incoming predicate
incoming_predicate_persons = defaultdict(set)

for subject, predicate, obj in graph:

    # Person appears as the OBJECT
    if str(obj) in labeled_persons:

        predicate_name = get_predicate_name(predicate)

        incoming_predicate_persons[predicate_name].add(str(obj))

In [24]:
incoming_summary = []

for predicate, persons in incoming_predicate_persons.items():

    incoming_summary.append({
        "Incoming Predicate": predicate,
        "Persons": len(persons),
        "Coverage (%)": round(
            len(persons) / len(labeled_persons) * 100,
            2
        )
    })

incoming_df = (
    pd.DataFrame(incoming_summary)
      .sort_values("Persons", ascending=False)
      .reset_index(drop=True)
)

incoming_df

,Incoming Predicate,Persons,Coverage (%)
0,author,124,70.45
1,member,102,57.95
2,isWorkedOnBy,98,55.68
3,editor,24,13.64
4,employs,5,2.84
5,head,5,2.84


In [25]:
# Side-by-side display function
def display_side_by_side(*dfs, titles=[]):
    html_str = '<div style="display: flex;">'
    for df, title in zip(dfs, titles):
        html_str += f'<div style="margin-right: 20px;">'
        html_str += f'<h3>{title}</h3>'
        html_str += df.to_html()
        html_str += '</div>'
    html_str += '</div>'
    display(HTML(html_str))

# Run this in your cell to see them side-by-side
display_side_by_side(predicate_df, incoming_df, titles=['Outgoing DF', 'Incoming DF'])


,Outgoing Predicate,Persons,Coverage (%)
0,affiliation,176,100.00
1,name,176,100.00
2,type,176,100.00
3,phone,176,100.00
4,fax,176,100.00
5,publication,124,70.45
6,photo,113,64.20
7,worksAtProject,77,43.75
8,homepage,72,40.91
,Incoming Predicate,Persons,Coverage (%)


### **Very Important**

If we comapre the two tables are we can make some important observations

Outgoing   publication → 70%  
Incoming   author      → 70%

Means what this tells is

Person  ----publication----- Publication    
Publication   ------author------Person

These are simply inverse relations. That means they describe the same information.

### <span style="color:#FF5733; font-weight:bold;">The main question now is: Which information around a person actually helps to identify their research group?</span>

- phone, name, type, fax will have 100% coverage but knowing this info will not help to predict the reserach group
- affiliation any way will be discarded because it is the lael/target
- publication, worksAtProject is important so lets explore on this more
- photo and homepage doesn't make sense so lets ignore it


# 2) Neighbourhood Exploration

## Publication

For example,

Personen(id2040instance) -----publication ------- Publikationen(id864instance)

and then we explore the everything attached to that Publikationen(id864instance).

In [26]:
# Collect all Publication nodes

publication_nodes = set()

for subject, predicate, obj in graph:

    if (
        get_predicate_name(predicate) == "type"
        and get_entity_name(obj) == "Publication"
    ):
        publication_nodes.add(str(subject))

print(f"Total Publication nodes: {len(publication_nodes)}")

Total Publication nodes: 1222


In [27]:
print("First 10 Publication nodes\n")

for i, publication in enumerate(sorted(publication_nodes)):

    print(get_entity_name(publication))

    if i == 9:
        break

First 10 Publication nodes

Publikationen(id0instance)
Publikationen(id1000instance)
Publikationen(id1001instance)
Publikationen(id1002instance)
Publikationen(id1003instance)
Publikationen(id1004instance)
Publikationen(id1005instance)
Publikationen(id1006instance)
Publikationen(id1007instance)
Publikationen(id1008instance)


In [28]:
example_publication = sorted(publication_nodes)[0]

print(example_publication)

http://www.aifb.uni-karlsruhe.de/Publikationen/viewPublikationOWL/id0instance


In [29]:
print("=" * 80)
print("Outgoing Triples")
print("=" * 80)

for subject, predicate, obj in graph:

    if str(subject) == example_publication:

        print_triple(subject, predicate, obj)

Outgoing Triples
Subject   : Publikationen(id0instance)
Predicate : type
Object    : InProceedings
--------------------------------------------------------------------------------
Subject   : Publikationen(id0instance)
Predicate : abstract
Object    : OntoWeb is an European Union IST-funded thematic network for Ontology-based information exchange for knowledge management and electronic commerce. The corresponding OntoWeb portal constitutes a Web-based research
information system that is driven by some of the technologies which it
reports about. In this paper, we present the core methodology underlying
the OntoWeb portal,viz. SEAL (SEmantic portAL). In particular, we describe
some of the core challenges that SEAL must meet. Because of the distributed nature of research information, SEAL has been developed as a methodology that integrates heterogeneous information from distributed resources.
Because of the complexity of the application domain, SEAL is based on
ontologies about research i

In [30]:
print("=" * 80)
print("Incoming Triples")
print("=" * 80)

for subject, predicate, obj in graph:

    if str(obj) == example_publication:

        print_triple(subject, predicate, obj)

Incoming Triples
Subject   : Forschungsgruppen(id3instance)
Predicate : publishes
Object    : Publikationen(id0instance)
--------------------------------------------------------------------------------
Subject   : Personen(id40instance)
Predicate : publication
Object    : Publikationen(id0instance)
--------------------------------------------------------------------------------
Subject   : Projekte(id10instance)
Predicate : projectInfo
Object    : Publikationen(id0instance)
--------------------------------------------------------------------------------
Subject   : Personen(id6instance)
Predicate : publication
Object    : Publikationen(id0instance)
--------------------------------------------------------------------------------
Subject   : Personen(id57instance)
Predicate : publication
Object    : Publikationen(id0instance)
--------------------------------------------------------------------------------
Subject   : Personen(id80instance)
Predicate : publication
Object    : Publikatione

In [31]:
publication_predicates = defaultdict(set)

for subject, predicate, obj in graph:

    if str(subject) in publication_nodes:

        publication_predicates[
            get_predicate_name(predicate)
        ].add(str(subject))

summary = []

for predicate, publications in publication_predicates.items():

    summary.append({
        "Outgoing Predicate": predicate,
        "Publications": len(publications),
        "Coverage (%)": round(
            len(publications) / len(publication_nodes) * 100,
            2
        )
    })

publication_outgoing_df = (
    pd.DataFrame(summary)
      .sort_values("Publications", ascending=False)
      .reset_index(drop=True)
)

publication_outgoing_df

,Outgoing Predicate,Publications,Coverage (%)
0,type,1222,100.00
1,year,1217,99.59
2,title,1217,99.59
3,author,1165,95.34
4,booktitle,761,62.27
5,month,751,61.46
6,isAbout,708,57.94
7,hasProject,576,47.14
8,pages,545,44.60
9,abstract,532,43.54


**Analysis**

We can make ike this analysis from the above table

| Information | Opinion |
|:------------|:--------|
| Research Topic (isAbout) | Important |
| Projects (hasProject) | Important |
| Conference (booktitle) | Important |
| Publication Count | ok |
| Publication Year | ok |
| Abstract | Important but too complicated |
| Title | ok but not useful directly |
| Pages | no |
| ISBN | no |
| Edition | no |
| Address | no |

### Research Topic (isAbout)

In [32]:
# Collect all Research Topic nodes
research_topic_nodes = set()

for subject, predicate, obj in graph:

    if (
        get_predicate_name(predicate) == "type"
        and get_entity_name(obj) == "ResearchTopic"
    ):
        research_topic_nodes.add(str(subject))

print(f"Total Research Topic nodes: {len(research_topic_nodes)}")

Total Research Topic nodes: 146


In [33]:
print("First 10 Research Topics\n")

for i, topic in enumerate(sorted(research_topic_nodes)):

    print(get_entity_name(topic))

    if i == 9:
        break

First 10 Research Topics

Forschungsgebiete(id100instance)
Forschungsgebiete(id101instance)
Forschungsgebiete(id102instance)
Forschungsgebiete(id103instance)
Forschungsgebiete(id104instance)
Forschungsgebiete(id105instance)
Forschungsgebiete(id106instance)
Forschungsgebiete(id107instance)
Forschungsgebiete(id108instance)
Forschungsgebiete(id109instance)


In [34]:
example_topic = sorted(research_topic_nodes)[0]

print("Example Research Topic:")
print(example_topic)

Example Research Topic:
http://www.aifb.uni-karlsruhe.de/Forschungsgebiete/viewForschungsgebietOWL/id100instance


In [35]:
print("="*80)
print("Outgoing Triples")
print("="*80)

for subject, predicate, obj in graph:

    if str(subject) == example_topic:

        print_triple(subject, predicate, obj)

Outgoing Triples
Subject   : Forschungsgebiete(id100instance)
Predicate : isWorkedOnBy
Object    : Personen(id2044instance)
--------------------------------------------------------------------------------
Subject   : Forschungsgebiete(id100instance)
Predicate : isWorkedOnBy
Object    : Personen(id13instance)
--------------------------------------------------------------------------------
Subject   : Forschungsgebiete(id100instance)
Predicate : isWorkedOnBy
Object    : Personen(id6instance)
--------------------------------------------------------------------------------
Subject   : Forschungsgebiete(id100instance)
Predicate : name
Object    : www systems
--------------------------------------------------------------------------------
Subject   : Forschungsgebiete(id100instance)
Predicate : dealtWithIn
Object    : Projekte(id6instance)
--------------------------------------------------------------------------------
Subject   : Forschungsgebiete(id100instance)
Predicate : dealtWithIn
Obje

In [36]:
print("="*80)
print("Incoming Triples")
print("="*80)

for subject, predicate, obj in graph:

    if str(obj) == example_topic:

        print_triple(subject, predicate, obj)

Incoming Triples
Subject   : Publikationen(id1061instance)
Predicate : isAbout
Object    : Forschungsgebiete(id100instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id152instance)
Predicate : isAbout
Object    : Forschungsgebiete(id100instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id1269instance)
Predicate : isAbout
Object    : Forschungsgebiete(id100instance)
--------------------------------------------------------------------------------
Subject   : Projekte(id58instance)
Predicate : isAbout
Object    : Forschungsgebiete(id100instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id1055instance)
Predicate : isAbout
Object    : Forschungsgebiete(id100instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id8instance)
Predicate : 

In [37]:
from collections import defaultdict

topic_predicates = defaultdict(set)

for subject, predicate, obj in graph:

    if str(subject) in research_topic_nodes:

        topic_predicates[
            get_predicate_name(predicate)
        ].add(str(subject))

summary = []

for predicate, topics in topic_predicates.items():

    summary.append({
        "Outgoing Predicate": predicate,
        "Research Topics": len(topics),
        "Coverage (%)": round(
            len(topics) / len(research_topic_nodes) * 100,
            2
        )
    })

topic_outgoing_df = (
    pd.DataFrame(summary)
      .sort_values("Research Topics", ascending=False)
      .reset_index(drop=True)
)

topic_outgoing_df

,Outgoing Predicate,Research Topics,Coverage (%)
0,name,146,100.00
1,type,146,100.00
2,http://www.aifb.uni-karlsruhe.de/WBS/dvr/owlto...,129,88.36
3,isWorkedOnBy,124,84.93
4,dealtWithIn,110,75.34


<span style="color:#FF5733; font-weight:bold;">By seeing this outgoing Predicates another question rises which is Do different research groups actually work on different research topics?</span>

#### Relationship between Research Groups and Research Topics

In [38]:
topic_name_map = {}

for subject, predicate, obj in graph:

    if (
        str(subject) in research_topic_nodes
        and get_predicate_name(predicate) == "name"
    ):

        topic_name_map[str(subject)] = str(obj)

In [39]:
list(topic_name_map.items())[:5]

[('http://www.aifb.uni-karlsruhe.de/Forschungsgebiete/viewForschungsgebietOWL/id77instance',
  'reconfigurability'),
 ('http://www.aifb.uni-karlsruhe.de/Forschungsgebiete/viewForschungsgebietOWL/id87instance',
  'systems engineering'),
 ('http://www.aifb.uni-karlsruhe.de/Forschungsgebiete/viewForschungsgebietOWL/id28instance',
  'decision support systems'),
 ('http://www.aifb.uni-karlsruhe.de/Forschungsgebiete/viewForschungsgebietOWL/id14instance',
  'constraint programming'),
 ('http://www.aifb.uni-karlsruhe.de/Forschungsgebiete/viewForschungsgebietOWL/id30instance',
  'evolutionary algorithms')]

In [40]:
person_topics = defaultdict(set)

for person in labeled_persons:

    # Find publications of this person
    publications = set()

    for subject, predicate, obj in graph:

        if (
            str(subject) == person
            and get_predicate_name(predicate) == "publication"
        ):
            publications.add(str(obj))

    # Find research topics of those publications
    for publication in publications:

        for subject, predicate, obj in graph:

            if (
                str(subject) == publication
                and get_predicate_name(predicate) == "isAbout"
            ):

                topic_uri = str(obj)

                if topic_uri in topic_name_map:

                    person_topics[person].add(topic_name_map[topic_uri])

In [41]:
rows = []

for person, topics in person_topics.items():

    rows.append({

        "Person": get_entity_name(person),

        "Number of Topics": len(topics),

        "Research Topics": sorted(topics)

    })

person_topic_df = pd.DataFrame(rows)

person_topic_df.head()

,Person,Number of Topics,Research Topics
0,Personen(id2058instance),15,"[Petri nets, business process analysis, busine..."
1,Personen(id99instance),5,"[development of knowledge management systems, ..."
2,Personen(id48instance),5,"[ant algorithms, computer architecture, evolut..."
3,Personen(id2064instance),2,"[e-business, electronic commerce]"
4,Personen(id2062instance),3,"[Computer Science, e-Learning, virtual organiz..."


In [42]:
complete_df_readable = complete_df.copy()

complete_df_readable["Person"] = (
    complete_df_readable["person"]
    .apply(get_entity_name)
)

complete_df_readable["Research Group"] = (
    complete_df_readable["label_affiliation"]
    .apply(get_entity_name)
)

person_topic_df = person_topic_df.merge(

    complete_df_readable[["Person", "Research Group"]],

    on="Person",

    how="left"

)

person_topic_df.head()

,Person,Number of Topics,Research Topics,Research Group
0,Personen(id2058instance),15,"[Petri nets, business process analysis, busine...",Forschungsgruppen(id1instance)
1,Personen(id99instance),5,"[development of knowledge management systems, ...",Forschungsgruppen(id1instance)
2,Personen(id48instance),5,"[ant algorithms, computer architecture, evolut...",Forschungsgruppen(id2instance)
3,Personen(id2064instance),2,"[e-business, electronic commerce]",Forschungsgruppen(id1instance)
4,Personen(id2062instance),3,"[Computer Science, e-Learning, virtual organiz...",Forschungsgruppen(id1instance)


In [43]:
print(f"Persons with topics: {len(person_topic_df)}")

person_topic_df.sample(10, random_state=42)

Persons with topics: 109


,Person,Number of Topics,Research Topics,Research Group
78,Personen(id90instance),30,"[Digital libraries, Multimedia-Engineering, On...",Forschungsgruppen(id3instance)
10,Personen(id2132instance),14,"[Grid Computing, Ontology Engineering, Ontolog...",Forschungsgruppen(id3instance)
4,Personen(id2062instance),3,"[Computer Science, e-Learning, virtual organiz...",Forschungsgruppen(id1instance)
84,Personen(id1954instance),14,"[Digital libraries, Petri nets, business proce...",Forschungsgruppen(id1instance)
64,Personen(id38instance),30,"[Aspect-Oriented Programming, Business models,...",Forschungsgruppen(id1instance)
68,Personen(id14instance),19,"[Digital libraries, Information Retrieval, Nat...",Forschungsgruppen(id3instance)
30,Personen(id12instance),4,"[knowledge management, m-business, mobile tech...",Forschungsgruppen(id1instance)
45,Personen(id2023instance),30,"[Artificial Intelligence, Digital libraries, N...",Forschungsgruppen(id3instance)
96,Personen(id2081instance),8,"[Organic Computing, e-Learning, efficient algo...",Forschungsgruppen(id2instance)
11,Personen(id2079instance),25,"[Information Retrieval, Multimedia Annotation ...",Forschungsgruppen(id3instance)


In [44]:
group_topics = {}

for group in person_topic_df["Research Group"].unique():

    topics = []

    group_df = person_topic_df[
        person_topic_df["Research Group"] == group
    ]

    for topic_list in group_df["Research Topics"]:

        topics.extend(topic_list)

    group_topics[group] = Counter(topics)

In [45]:
for group, counter in group_topics.items():

    print("=" * 80)
    print(group)
    print("=" * 80)

    for topic, count in counter.most_common(15):

        print(f"{topic:35} {count}")

    print()

Forschungsgruppen(id1instance)
Petri nets                          15
e-Learning                          13
business process management         11
business process modelling          11
Web Services                        10
workflow management                 9
modeling                            7
teaching modules                    7
www systems                         7
Digital libraries                   7
learning modules                    6
office information systems          6
electronic commerce                 6
m-business                          6
database systems                    5

Forschungsgruppen(id2instance)
evolutionary algorithms             9
efficient algorithms                9
e-Learning                          8
parallel algorithms                 7
ant algorithms                      6
mobile technologies                 6
reconfigurability                   5
reconfigurable mesh                 5
information systems                 5
learning modules    

### Project

In [46]:
# Collect all Project nodes
project_nodes = set()

for subject, predicate, obj in graph:

    if (
        get_predicate_name(predicate) == "type"
        and get_entity_name(obj) == "Project"
    ):
        project_nodes.add(str(subject))

print(f"Total Project nodes: {len(project_nodes)}")

Total Project nodes: 78


In [47]:
print("First 10 Project nodes\n")

for i, project in enumerate(sorted(project_nodes)):

    print(get_entity_name(project))

    if i == 9:
        break

First 10 Project nodes

Projekte(id10instance)
Projekte(id11instance)
Projekte(id12instance)
Projekte(id13instance)
Projekte(id14instance)
Projekte(id15instance)
Projekte(id16instance)
Projekte(id17instance)
Projekte(id18instance)
Projekte(id19instance)


In [48]:
example_project = sorted(project_nodes)[0]

print("Example Project:")
print(example_project)

Example Project:
http://www.aifb.uni-karlsruhe.de/Projekte/viewProjektOWL/id10instance


In [49]:
print("=" * 80)
print("Outgoing Triples")
print("=" * 80)

for subject, predicate, obj in graph:

    if str(subject) == example_project:

        print_triple(subject, predicate, obj)

Outgoing Triples
Subject   : Projekte(id10instance)
Predicate : projectInfo
Object    : Publikationen(id714instance)
--------------------------------------------------------------------------------
Subject   : Projekte(id10instance)
Predicate : isAbout
Object    : Forschungsgebiete(id69instance)
--------------------------------------------------------------------------------
Subject   : Projekte(id10instance)
Predicate : projectInfo
Object    : Publikationen(id718instance)
--------------------------------------------------------------------------------
Subject   : Projekte(id10instance)
Predicate : projectInfo
Object    : Publikationen(id0instance)
--------------------------------------------------------------------------------
Subject   : Projekte(id10instance)
Predicate : projectInfo
Object    : Publikationen(id2instance)
--------------------------------------------------------------------------------
Subject   : Projekte(id10instance)
Predicate : projectInfo
Object    : Publikatione

In [50]:
print("=" * 80)
print("Incoming Triples")
print("=" * 80)

for subject, predicate, obj in graph:

    if str(obj) == example_project:

        print_triple(subject, predicate, obj)

Incoming Triples
Subject   : Publikationen(id716instance)
Predicate : hasProject
Object    : Projekte(id10instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id153instance)
Predicate : hasProject
Object    : Projekte(id10instance)
--------------------------------------------------------------------------------
Subject   : Forschungsgebiete(id70instance)
Predicate : dealtWithIn
Object    : Projekte(id10instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id174instance)
Predicate : hasProject
Object    : Projekte(id10instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id721instance)
Predicate : hasProject
Object    : Projekte(id10instance)
--------------------------------------------------------------------------------
Subject   : Publikationen(id164instance)
Predicate : hasProject
Object    : Pro

In [51]:
project_predicates = defaultdict(set)

for subject, predicate, obj in graph:

    if str(subject) in project_nodes:

        project_predicates[
            get_predicate_name(predicate)
        ].add(str(subject))

summary = []

for predicate, projects in project_predicates.items():

    summary.append({

        "Outgoing Predicate": predicate,

        "Projects": len(projects),

        "Coverage (%)": round(
            len(projects) / len(project_nodes) * 100,
            2
        )
    })

project_outgoing_df = (
    pd.DataFrame(summary)
      .sort_values("Projects", ascending=False)
      .reset_index(drop=True)
)

project_outgoing_df

,Outgoing Predicate,Projects,Coverage (%)
0,type,78,100.00
1,name,75,96.15
2,carriedOutBy,74,94.87
3,member,73,93.59
4,isAbout,71,91.03
5,financedBy,58,74.36
6,projectInfo,57,73.08
7,homepage,57,73.08


In [52]:
incoming_project_predicates = defaultdict(set)

for subject, predicate, obj in graph:

    if str(obj) in project_nodes:

        incoming_project_predicates[
            get_predicate_name(predicate)
        ].add(str(obj))

summary = []

for predicate, projects in incoming_project_predicates.items():

    summary.append({

        "Incoming Predicate": predicate,

        "Projects": len(projects),

        "Coverage (%)": round(
            len(projects) / len(project_nodes) * 100,
            2
        )
    })

project_incoming_df = (
    pd.DataFrame(summary)
      .sort_values("Projects", ascending=False)
      .reset_index(drop=True)
)

project_incoming_df

,Incoming Predicate,Projects,Coverage (%)
0,carriesOut,74,94.87
1,dealtWithIn,71,91.03
2,finances,61,78.21
3,hasProject,57,73.08
4,worksAtProject,49,62.82


In [53]:
project_name_map = {}

for subject, predicate, obj in graph:

    if (
        str(subject) in project_nodes
        and get_predicate_name(predicate) == "name"
    ):

        project_name_map[str(subject)] = str(obj)

print(f"Project names collected: {len(project_name_map)}")

list(project_name_map.items())[:10]

Project names collected: 75


[('http://www.aifb.uni-karlsruhe.de/Projekte/viewProjektOWL/id18instance',
  'PEdit - Programmierumgebung fÃ¼r parallele Systeme'),
 ('http://www.aifb.uni-karlsruhe.de/Projekte/viewProjektOWL/id53instance',
  'Arbeit@VU - Arbeit@VU - Gestaltung der Arbeit in virtuellen Unternehmen'),
 ('http://www.aifb.uni-karlsruhe.de/Projekte/viewProjektOWL/id29instance',
  'NUKATH - Notebook UniversitÃ¤t  Karlsruhe (TH)'),
 ('http://www.aifb.uni-karlsruhe.de/Projekte/viewProjektOWL/id63instance',
  'KIM - Karlsruher Integriertes Informations-Management'),
 ('http://www.aifb.uni-karlsruhe.de/Projekte/viewProjektOWL/id67instance',
  'Semantic MediaWiki - Wikipedia und das Semantic Web'),
 ('http://www.aifb.uni-karlsruhe.de/Projekte/viewProjektOWL/id35instance',
  'Dot.Kom - Designing adaptive infOrmation exTraction for KnOwledge Management'),
 ('http://www.aifb.uni-karlsruhe.de/Projekte/viewProjektOWL/id25instance',
  'ITSAM - IT-UnterstÃ¼tzung fÃ¼r das Asset Management'),
 ('http://www.aifb.uni-karls

In [54]:
person_projects = defaultdict(set)

for person in labeled_persons:

    publications = set()

    # Find publications of the person
    for subject, predicate, obj in graph:

        if (
            str(subject) == person
            and get_predicate_name(predicate) == "publication"
        ):

            publications.add(str(obj))

    # Find projects linked to those publications
    for publication in publications:

        for subject, predicate, obj in graph:

            if (
                str(subject) == publication
                and get_predicate_name(predicate) == "hasProject"
            ):

                project_uri = str(obj)

                if project_uri in project_name_map:

                    person_projects[person].add(
                        project_name_map[project_uri]
                    )

In [55]:
rows = []

for person, projects in person_projects.items():

    rows.append({

        "Person": get_entity_name(person),

        "Number of Projects": len(projects),

        "Projects": sorted(projects)

    })

person_project_df = pd.DataFrame(rows)

person_project_df.head()

,Person,Number of Projects,Projects
0,Personen(id2058instance),2,"[FIS-I - Fachinformationssystem Informatik, Fl..."
1,Personen(id99instance),1,[HARMONISE - Studie zur Harmonisierung von IT ...
2,Personen(id48instance),1,[OptRek - Optimierung auf rekonfigurierbaren R...
3,Personen(id2062instance),3,[Arbeit@VU - Arbeit@VU - Gestaltung der Arbeit...
4,Personen(id80instance),13,[(KA)2 - Knowledge Annotation Initiative of th...


In [56]:
person_project_df = person_project_df.merge(

    complete_df_readable[
        ["Person", "Research Group"]
    ],

    on="Person",

    how="left"

)

person_project_df.head()

,Person,Number of Projects,Projects,Research Group
0,Personen(id2058instance),2,"[FIS-I - Fachinformationssystem Informatik, Fl...",Forschungsgruppen(id1instance)
1,Personen(id99instance),1,[HARMONISE - Studie zur Harmonisierung von IT ...,Forschungsgruppen(id1instance)
2,Personen(id48instance),1,[OptRek - Optimierung auf rekonfigurierbaren R...,Forschungsgruppen(id2instance)
3,Personen(id2062instance),3,[Arbeit@VU - Arbeit@VU - Gestaltung der Arbeit...,Forschungsgruppen(id1instance)
4,Personen(id80instance),13,[(KA)2 - Knowledge Annotation Initiative of th...,Forschungsgruppen(id3instance)


In [57]:
from collections import Counter

group_projects = {}

for group in person_project_df["Research Group"].unique():

    projects = []

    group_df = person_project_df[
        person_project_df["Research Group"] == group
    ]

    for project_list in group_df["Projects"]:

        projects.extend(project_list)

    group_projects[group] = Counter(projects)

In [58]:
for group, counter in group_projects.items():

    print("=" * 80)
    print(group)
    print("=" * 80)

    for project, count in counter.most_common(15):

        print(f"{project:45} {count}")

    print()

Forschungsgruppen(id1instance)
FIS-I - Fachinformationssystem Informatik     5
FlexAn - Flexible Abrechnungssysteme          5
MoMaTIK - Mobiles Marketing - Technologie Informationszentrum Uni Karlsruhe 5
ViKar - Virtueller Hochschulverbund Karlsruhe 4
HARMONISE - Studie zur Harmonisierung von IT Zertifizierungssystemen fÃ¼r IT Professionals in Europa 3
SemIPort - Semantic Methods and Tools for Information Portals 3
VIROR - Virtuelle UniversitÃ¤t Oberrhein      3
Arbeit@VU - Arbeit@VU - Gestaltung der Arbeit in virtuellen Unternehmen 2
eSkills Cert - eSkills Certification in Europe 2
Graduiertenkolleg IME - Informationswirtschaft und Market Engineering 2
SEKT - Semantically Enabled Knowledge Technologies 2
CONsense - Cooperatives Wissensmanagement in Virtuellen Organisationen 1
KIM - Karlsruher Integriertes Informations-Management 1

Forschungsgruppen(id2instance)
NUKATH - Notebook UniversitÃ¤t  Karlsruhe (TH) 7
OptRek - Optimierung auf rekonfigurierbaren Rechensystemen 4
AntAlg - Amei

### **Conclusion**

| **Research Topics** | **Projects** |
|:--------------------|:-------------|
| Independent research interest | Often organizational information |
| Obtained through publications | Connected to research groups |
| Good semantic feature | Possible leakage |
| Generalizable | Dataset specific |
| Easier to explain | May overfit |

By seeing the project information we can tell
Research Topics -->  are the true semantic information.

Projects -->  are more organizational information.

# 3) Candidate Feature Discovery

This tells about the features we considered, why we considered them, and why we kept or rejected each one.

## 3.1 — Candidate Features from Person

| Information    | Coverage | Initial Thought     |
| -------------- | -------- | ------------------- |
| phone          | 100%     | Probably not useful |
| fax            | 100%     | Probably not useful |
| homepage       | 41%      | Maybe useful        |
| publication    | 70%      | Very interesting    |
| worksAtProject | 44%      | Interesting         |
| photo          | 64%      | Probably not useful |
| affiliation    | 100%     | **Target leakage**  |

### 3.2 — Candidate Features from Publication

| Information            | Initial Thought      |
| ---------------------- | -------------------- |
| title                  | Metadata             |
| year                   | Possible feature     |
| conference (booktitle) | Interesting          |
| research topics        | Very promising       |
| project                | Use Carefully        |
| abstract               | Rich but too complex |
| pages                  | Not useful           |
| ISBN                   | Not useful           |

**Note**: WorksAtProject and Projects are not same, WorksAtProject comes from direct Person and Projects comes from Publication 

#### 3.3 — Candidate Features from Research Topic

We discovered:

* topics clearly differ across research groups,
* topic names carry semantic meaning,
* topics are linked through publications.

This is probably our strongest candidate.

#### 3.4 — Candidate Features from Project

We discovered something important here:

Project --> carriedOutBy --> Research Group

That means Projects may contain **target leakage**.

## Candidate Feature Summary

| Candidate Feature | Source Entity | Coverage | Predictive Potential | Leakage Risk | Decision      |
| ----------------- | ------------- | -------- | -------------------- | ------------ | ------------- |
| Homepage          | Person        | 41%      | Low                  | None         | Maybe         |
| Publication Count | Person        | 70%      | Medium               | None         | Keep          |
| worksAtProject    | Person        | 44%      | Medium               | None         | Keep          |
| Affiliation       | Person        | 100%     | Very High            | **High**     | Reject        |
| Research Topics   | Publication   | 58%      | Very High            | None         | Keep          |
| Conference        | Publication   | 62%      | Medium               | None         | Maybe         |
| Publication Year  | Publication   | 99%      | Low                  | None         | Maybe         |
| Abstract          | Publication   | 43%      | High                 | None         | Reject        |
| Projects          | Publication   | 47%      | High                 | Medium       | Use Carefully |

# 4) Feature Design Decisions

This tells us about how can we represent Research Topics numerically

###  Target Variable (which is Research Group)

4 classes ->  Taken from label_affiliation column

### Person-level Features

<pre>
Homepage ───┬────── URL  
            └────── Yes/No 
</pre>

*Decision: Binary i.e Yes/No*

<pre>
Publication Count ───┬────── Publication IDs
                     ├────── Publication Count
                     └────── Binary Publication Columns
</pre>

*Decision: Publication Count*

<pre>
WorksAtProject ───┬────── Count
                  ├────── Project IDs
                  └────── Binary
</pre>

*Decision: Count*

### Publication Features

<pre>
Research Topics ───┬────── Topic Count
                   ├────── Store in english
                   └────── One-hot encoding
</pre>

*Decision: One-hot encoding*

<pre>
Conference ───┬────── Full conference name
              ├────── One-hot encoding
              └────── Ignore
</pre>

*Decision:*

<pre>
Publication Year ───────── Integer

</pre>

*Decision: Integer*

<pre>
Projects ───┬────── Ignore
            ├────── Use Project Count
            └────── One-hot encode
</pre>

*Decision:*

### Final Conclusion

| Feature           | Encoding           | Why this encoding?                                             |
| ----------------- | ------------------ | -------------------------------------------------------------- |
| Homepage          | Binary             | Only indicates presence/absence                                |
| Publication Count | Integer            | Preserves quantity without creating hundreds of sparse columns |
| WorksAtProject    | Count              | Represents the level of project involvement                    |
| Research Topics   | Multi-hot          | A person can work on multiple topics simultaneously            |
| Conference        | (Decision pending) | Where the research was published.                              |
| Publication Year  | Integer            | Temporal information is naturally numeric                      |
| Projects          | (Decision pending) | To be justified after considering leakage                      |


## Conference Analysis

Questions to answer

For every research group:  
- What are the top conference/booktitles?  
- Do different groups publish in different venues?  
- Is there a lot of overlap?  
- Are some conferences unique to one group?

Outcomes:  

Very different conferences  --> Keep  
Mostly identical conferences --> Rejeect 

In [59]:
# Build Person → Conference Mapping
from collections import defaultdict

# Stores conferences for each person
person_conferences = defaultdict(set)

for person in labeled_persons:

    publications = set()

    # Find publications of this person
    for subject, predicate, obj in graph:

        if (
            str(subject) == person
            and get_predicate_name(predicate) == "publication"
        ):
            publications.add(str(obj))

    # Find conferences (booktitle) of those publications
    for publication in publications:

        for subject, predicate, obj in graph:

            if (
                str(subject) == publication
                and get_predicate_name(predicate) == "booktitle"
            ):

                person_conferences[person].add(str(obj))

print(f"Persons with conference information: {len(person_conferences)}")

Persons with conference information: 116


In [60]:
rows = []

for person, conferences in person_conferences.items():

    rows.append({

        "Person": get_entity_name(person),

        "Number of Conferences": len(conferences),

        "Conferences": sorted(conferences)

    })

person_conference_df = pd.DataFrame(rows)

person_conference_df.head()

,Person,Number of Conferences,Conferences
0,Personen(id2058instance),15,[37th Hawaii International Conference on Syste...
1,Personen(id99instance),8,[5th International Conference on Knowledge Man...
2,Personen(id48instance),3,"[Evolutionary Multi-Criterion Optimization, Se..."
3,Personen(id2064instance),4,[Proceedings of the 6th International Business...
4,Personen(id2062instance),9,"[Building the Knowledge Economy: Issues, Appli..."


In [61]:
# Merge with Research Groups
person_conference_df = person_conference_df.merge(

    complete_df_readable[["Person", "Research Group"]],

    on="Person",

    how="left"

)

person_conference_df.head()

,Person,Number of Conferences,Conferences,Research Group
0,Personen(id2058instance),15,[37th Hawaii International Conference on Syste...,Forschungsgruppen(id1instance)
1,Personen(id99instance),8,[5th International Conference on Knowledge Man...,Forschungsgruppen(id1instance)
2,Personen(id48instance),3,"[Evolutionary Multi-Criterion Optimization, Se...",Forschungsgruppen(id2instance)
3,Personen(id2064instance),4,[Proceedings of the 6th International Business...,Forschungsgruppen(id1instance)
4,Personen(id2062instance),9,"[Building the Knowledge Economy: Issues, Appli...",Forschungsgruppen(id1instance)


In [62]:
print(f"Persons with conference information: {len(person_conference_df)}")

person_conference_df.sample(10, random_state=42)

Persons with conference information: 116


,Person,Number of Conferences,Conferences,Research Group
83,Personen(id2042instance),18,"[CRM-Erfolgsfaktor Kundenorientierung, CRM-Sys...",Forschungsgruppen(id1instance)
4,Personen(id2062instance),9,"[Building the Knowledge Economy: Issues, Appli...",Forschungsgruppen(id1instance)
42,Personen(id2069instance),1,[Tagungsband des 13. Workshops Algorithmen und...,Forschungsgruppen(id1instance)
40,Personen(id2053instance),9,[AAAI Fall Symposium On Semantic Web for Colla...,Forschungsgruppen(id3instance)
10,Personen(id11instance),20,[AAAI 2000/IAAI 2000 - Proceedings of the 17th...,Forschungsgruppen(id3instance)
47,Personen(id2078instance),15,[6th IEEE International Workshop on Policies f...,Forschungsgruppen(id3instance)
110,Personen(id2043instance),3,"[Mobile Economy - Transaktionen, Prozesse, Anw...",Forschungsgruppen(id1instance)
36,Personen(id43instance),2,[IADIS International Conference e-Commerce 200...,Forschungsgruppen(id4instance)
70,Personen(id2084instance),27,[19th Workshop on (Constraint) Logic Programmi...,Forschungsgruppen(id3instance)
11,Personen(id2132instance),7,[4th International Semantic Web Conference (IS...,Forschungsgruppen(id3instance)


In [63]:
group_conferences = {}

for group in person_conference_df["Research Group"].unique():

    conferences = []

    group_df = person_conference_df[
        person_conference_df["Research Group"] == group
    ]

    for conference_list in group_df["Conferences"]:

        conferences.extend(conference_list)

    group_conferences[group] = Counter(conferences)

In [64]:
for group, counter in group_conferences.items():

    print("=" * 80)
    print(group)
    print("=" * 80)

    for conference, count in counter.most_common(15):

        print(f"{conference[:70]:70} {count}")

    print()

Forschungsgruppen(id1instance)
Tagungsband des 13. Workshops Algorithmen und Werkzeuge fÃ¼r Petri-Net 9
EMISA 2006: Methoden, Konzepte und Technologien fÃ¼r die Entwicklung v 6
Business Information Systems, Proceedings of 7th BIS 2004              5
Proceedings Informatik 2003 - Innovative Informatikanwendungen, GI-Jah 5
Wirtschaftsinformatik 2003: Medien - MÃ¤rkte - MobilitÃ¤t              5
INFORMATIK 2004 - Informatik verbindet, GI-Jahrestagung, Ulm           4
Proceeding of the Workshop on Semantics for Business Process Managemen 4
Mobile Economy - Transaktionen, Prozesse, Anwendungen und Dienste      4
Mobile Informationssysteme - Potentiale, Hindernisse, Einsatz. Proceed 4
Perspektiven des Mobile Business - Wissenschaft und Praxis im Dialog   4
Proceedings of the IEEE International Conference on e-Business Enginee 4
8th IASTED International Conference on Computers and Advanced Technolo 3
Proc. 2nd IEEE Conference on e-Technology, e-Commerce and e-Services   3
Proceedings of the 1

So finally Conference gives publication venue information, which is slightly different from research topic information.

So we will keep this and for ecoding we do: **Multi-hot Encoding**

## Project Leakage Analysis

Questions to answer

- How severe is the leakage?  
- Does each project belong to exactly one research group?  
- Or are projects shared among groups?

Outcomes:  

Every project belongs to one research group only.  --> Projects become almost a label. **Dangerous**  
Projects are shared across many groups.   --> Projects become meaningful  **Multi-hot Encoding**

In [65]:
project_groups = defaultdict(set)

for subject, predicate, obj in graph:

    # Project carried out by Research Group
    if get_predicate_name(predicate) == "carriedOutBy":

        project = get_entity_name(subject)
        group = get_entity_name(obj)

        project_groups[project].add(group)

In [66]:
rows = []

for project, groups in project_groups.items():

    rows.append({

        "Project": project,

        "Number of Research Groups": len(groups),

        "Research Groups": sorted(groups)

    })

project_group_df = pd.DataFrame(rows)

project_group_df.head(10)

,Project,Number of Research Groups,Research Groups
0,Projekte(id45instance),1,[Forschungsgruppen(id2instance)]
1,Projekte(id43instance),1,[Forschungsgruppen(id3instance)]
2,Projekte(id41instance),1,[Forschungsgruppen(id2instance)]
3,Projekte(id19instance),3,"[Forschungsgruppen(id1instance), Forschungsgru..."
4,Projekte(id75instance),1,[Forschungsgruppen(id3instance)]
5,Projekte(id2instance),1,[Forschungsgruppen(id4instance)]
6,Projekte(id29instance),1,[Forschungsgruppen(id2instance)]
7,Projekte(id26instance),1,[Forschungsgruppen(id3instance)]
8,Projekte(id65instance),1,[Forschungsgruppen(id2instance)]
9,Projekte(id3instance),1,[Forschungsgruppen(id3instance)]


In [67]:
project_group_df["Number of Research Groups"].value_counts().sort_index()

Number of Research Groups
1    70
2     3
3     1
Name: count, dtype: int64

In [68]:
project_group_df[
    project_group_df["Number of Research Groups"] > 1
]

,Project,Number of Research Groups,Research Groups
3,Projekte(id19instance),3,"[Forschungsgruppen(id1instance), Forschungsgru..."
32,Projekte(id63instance),2,"[Forschungsgruppen(id1instance), Forschungsgru..."
33,Projekte(id31instance),2,"[Forschungsgruppen(id1instance), Forschungsgru..."
42,Projekte(id38instance),2,"[Forschungsgruppen(id2instance), Forschungsgru..."


If we calculate the percentages.

* 70/74 ≈ 94.6% of projects belong to exactly one research group.
* 4/74 ≈ 5.4% are shared between multiple groups.


Model will not learn the relationship between research interests and research groups. It's almost memorizing the answer.

We should **NOT one-hot encode Project IDs.** Instead we can keep Number of Projects

# 5) RDF → DataFrame

1 Create Base DataFrame, which is person and target

2 Add Simple Numeric Features like Homepage Feature, Publication Count, WorksAtProject Count, Number of Projects

3 Publication Year

4 Add Research Topic Features

5 Add Conference Features

6 Final Feature Matrix

7 Validate Feature Matrix

### 5.1 Create Base DataFrame

In [69]:
# Create the base feature dataframe
feature_df = complete_df_readable[["Person", "Research Group"]].copy()

feature_df.head()

,Person,Research Group
0,Personen(id1855instance),Forschungsgruppen(id1instance)
1,Personen(id1909instance),Forschungsgruppen(id1instance)
2,Personen(id2040instance),Forschungsgruppen(id3instance)
3,Personen(id46instance),Forschungsgruppen(id2instance)
4,Personen(id3instance),Forschungsgruppen(id2instance)


In [70]:
print(feature_df.isnull().sum())

Person            0
Research Group    0
dtype: int64


### 6.2) Add Simple Numeric Features

In [72]:
def has_homepage(person_name):
    """
    Returns:
        1 -> Person has a homepage
        0 -> No homepage
    """

    # Convert readable name back to URI
    person_uri = complete_df_readable.loc[
        complete_df_readable["Person"] == person_name,
        "person"
    ].iloc[0]

    for subject, predicate, obj in graph:

        if (
            str(subject) == person_uri
            and get_predicate_name(predicate) == "homepage"
        ):
            return 1

    return 0

In [73]:
feature_df["Homepage"] = feature_df["Person"].apply(has_homepage)

feature_df.head()

,Person,Research Group,Homepage
0,Personen(id1855instance),Forschungsgruppen(id1instance),1
1,Personen(id1909instance),Forschungsgruppen(id1instance),0
2,Personen(id2040instance),Forschungsgruppen(id3instance),0
3,Personen(id46instance),Forschungsgruppen(id2instance),1
4,Personen(id3instance),Forschungsgruppen(id2instance),1


In [74]:
print(feature_df["Homepage"].value_counts())

Homepage
0    104
1     72
Name: count, dtype: int64


In [75]:
def get_publication_count(person_name):
    """
    Counts the number of publications of a person.
    """

    person_uri = complete_df_readable.loc[
        complete_df_readable["Person"] == person_name,
        "person"
    ].iloc[0]

    count = 0

    for subject, predicate, obj in graph:

        if (
            str(subject) == person_uri
            and get_predicate_name(predicate) == "publication"
        ):
            count += 1

    return count

In [76]:
feature_df["Publication Count"] = (
    feature_df["Person"]
    .apply(get_publication_count)
)

feature_df.head()

,Person,Research Group,Homepage,Publication Count
0,Personen(id1855instance),Forschungsgruppen(id1instance),1,0
1,Personen(id1909instance),Forschungsgruppen(id1instance),0,0
2,Personen(id2040instance),Forschungsgruppen(id3instance),0,6
3,Personen(id46instance),Forschungsgruppen(id2instance),1,6
4,Personen(id3instance),Forschungsgruppen(id2instance),1,9


In [77]:
feature_df["Publication Count"].describe()

count    176.000000
mean      14.693182
std       30.030026
min        0.000000
25%        0.000000
50%        4.000000
75%       14.000000
max      213.000000
Name: Publication Count, dtype: float64

In [78]:
def get_works_at_project_count(person_name):
    """
    Counts the number of worksAtProject relationships.
    """

    person_uri = complete_df_readable.loc[
        complete_df_readable["Person"] == person_name,
        "person"
    ].iloc[0]

    count = 0

    for subject, predicate, obj in graph:

        if (
            str(subject) == person_uri
            and get_predicate_name(predicate) == "worksAtProject"
        ):
            count += 1

    return count

In [79]:
feature_df["WorksAtProject Count"] = (
    feature_df["Person"]
    .apply(get_works_at_project_count)
)

feature_df.head()

,Person,Research Group,Homepage,Publication Count,WorksAtProject Count
0,Personen(id1855instance),Forschungsgruppen(id1instance),1,0,0
1,Personen(id1909instance),Forschungsgruppen(id1instance),0,0,0
2,Personen(id2040instance),Forschungsgruppen(id3instance),0,6,0
3,Personen(id46instance),Forschungsgruppen(id2instance),1,6,2
4,Personen(id3instance),Forschungsgruppen(id2instance),1,9,2


In [80]:
feature_df["WorksAtProject Count"].describe()

count    176.000000
mean       1.102273
std        2.162220
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max       16.000000
Name: WorksAtProject Count, dtype: float64

In [81]:
def get_number_of_projects(person_name):
    """
    Counts the unique projects connected to a person's publications.
    """

    person_uri = complete_df_readable.loc[
        complete_df_readable["Person"] == person_name,
        "person"
    ].iloc[0]

    publications = set()

    for subject, predicate, obj in graph:

        if (
            str(subject) == person_uri
            and get_predicate_name(predicate) == "publication"
        ):
            publications.add(str(obj))

    projects = set()

    for publication in publications:

        for subject, predicate, obj in graph:

            if (
                str(subject) == publication
                and get_predicate_name(predicate) == "hasProject"
            ):
                projects.add(str(obj))

    return len(projects)

In [82]:
feature_df["Number of Projects"] = (
    feature_df["Person"]
    .apply(get_number_of_projects)
)

feature_df.head()

,Person,Research Group,Homepage,Publication Count,WorksAtProject Count,Number of Projects
0,Personen(id1855instance),Forschungsgruppen(id1instance),1,0,0,0
1,Personen(id1909instance),Forschungsgruppen(id1instance),0,0,0,0
2,Personen(id2040instance),Forschungsgruppen(id3instance),0,6,0,6
3,Personen(id46instance),Forschungsgruppen(id2instance),1,6,2,1
4,Personen(id3instance),Forschungsgruppen(id2instance),1,9,2,2


In [83]:
feature_df["Number of Projects"].describe()

count    176.000000
mean       2.818182
std        4.961671
min        0.000000
25%        0.000000
50%        0.500000
75%        3.000000
max       27.000000
Name: Number of Projects, dtype: float64

In [84]:
feature_df.head(10)

,Person,Research Group,Homepage,Publication Count,WorksAtProject Count,Number of Projects
0,Personen(id1855instance),Forschungsgruppen(id1instance),1,0,0,0
1,Personen(id1909instance),Forschungsgruppen(id1instance),0,0,0,0
2,Personen(id2040instance),Forschungsgruppen(id3instance),0,6,0,6
3,Personen(id46instance),Forschungsgruppen(id2instance),1,6,2,1
4,Personen(id3instance),Forschungsgruppen(id2instance),1,9,2,2
5,Personen(id1842instance),Forschungsgruppen(id1instance),1,0,0,0
6,Personen(id1915instance),Forschungsgruppen(id1instance),0,2,0,0
7,Personen(id1992instance),Forschungsgruppen(id1instance),0,0,0,0
8,Personen(id1966instance),Forschungsgruppen(id1instance),1,8,0,0
9,Personen(id2039instance),Forschungsgruppen(id1instance),0,14,0,1


### 6.3) Publication Year

In [85]:
# ============================================================================
# Publication URI -> Publication Year
# ============================================================================

publication_year_map = {}

for subject, predicate, obj in graph:

    if get_predicate_name(predicate) == "year":

        publication_year_map[str(subject)] = int(obj)

In [86]:
# ============================================================================
# Compute Average Publication Year for each Person
# ============================================================================

publication_years = []

for person in feature_df["Person"]:

    # Convert readable person name back to original URI
    person_uri = complete_df_readable.loc[
        complete_df_readable["Person"] == person,
        "person"
    ].iloc[0]

    years = []

    # Find this person's publications
    for subject, predicate, obj in graph:

        if (
            str(subject) == person_uri
            and get_predicate_name(predicate) == "publication"
        ):

            publication_uri = str(obj)

            if publication_uri in publication_year_map:
                years.append(publication_year_map[publication_uri])

    # Average publication year
    if len(years) == 0:
        publication_years.append(0)
    else:
        publication_years.append(round(sum(years) / len(years), 2))

In [87]:
# ============================================================================
# Add Publication Year Feature
# ============================================================================

feature_df["Publication Year"] = publication_years

### 6.4) Add Research Topic Features

In [88]:
# Collect all unique research topics

all_topics = set()

for topics in person_topic_df["Research Topics"]:

    for topic in topics:

        all_topics.add(topic)

all_topics = sorted(all_topics)

print(f"Total unique research topics: {len(all_topics)}")

print("\nFirst 10 topics:\n")

for topic in all_topics[:10]:
    print(topic)

Total unique research topics: 127

First 10 topics:

Artificial Intelligence
Aspect-Oriented Programming
Business models
Computational Biology
Computer Science
Courseware-Engineering
Customer Relationship Management
Digital libraries
EMS problems
FPGA


In [89]:
topic_features = pd.DataFrame()

topic_features["Person"] = person_topic_df["Person"]

In [90]:
topic_features.head()

,Person
0,Personen(id2058instance)
1,Personen(id99instance)
2,Personen(id48instance)
3,Personen(id2064instance)
4,Personen(id2062instance)


In [91]:
def clean_column_name(name, prefix=""):
    """
    Cleans feature names to make them suitable as DataFrame columns.

    Example:
    Artificial Intelligence
        ->
    Topic_Artificial_Intelligence
    """

    # Remove leading/trailing spaces
    name = name.strip()

    # Replace spaces with underscore
    name = name.replace(" ", "_")

    # Remove all non-alphanumeric characters except underscore
    name = re.sub(r"[^A-Za-z0-9_]", "", name)

    # Remove duplicate underscores
    name = re.sub(r"_+", "_", name)

    return prefix + name

In [92]:
# ============================================================================
# Build Multi-hot Topic Matrix
# ============================================================================

topic_matrix = {}

for topic in all_topics:

    clean_name = clean_column_name(topic, prefix="Topic_")

    topic_matrix[clean_name] = (

        person_topic_df["Research Topics"]

        .apply(lambda topics: int(topic in topics))

    )

topic_features = pd.concat(

    [

        person_topic_df[["Person"]],

        pd.DataFrame(topic_matrix)

    ],

    axis=1

)

In [93]:
topic_features.shape

(109, 127)

So only 109 rows → 109 persons who have at least one research topic.

In [94]:
topic_features.head()

,Person,Topic_Artificial_Intelligence,Topic_AspectOriented_Programming,Topic_Business_models,Topic_Computational_Biology,Topic_Computer_Science,Topic_CoursewareEngineering,Topic_Customer_Relationship_Management,Topic_Digital_libraries,Topic_EMS_problems,...,Topic_text_mining,Topic_theoretic_computer_science,Topic_topology_in_computer_science,Topic_tree_automata,Topic_usability_engineering,Topic_virtual_organizations,Topic_virtual_university,Topic_webbased_Learning,Topic_workflow_management,Topic_www_systems
0,Personen(id2058instance),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
1,Personen(id99instance),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Personen(id48instance),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Personen(id2064instance),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Personen(id2062instance),0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [95]:
feature_df = feature_df.merge(

    topic_features,

    on="Person",

    how="left"

)

In [96]:
feature_df.shape

(176, 133)

In [97]:
feature_df.head()

,Person,Research Group,Homepage,Publication Count,WorksAtProject Count,Number of Projects,Publication Year,Topic_Artificial_Intelligence,Topic_AspectOriented_Programming,Topic_Business_models,...,Topic_text_mining,Topic_theoretic_computer_science,Topic_topology_in_computer_science,Topic_tree_automata,Topic_usability_engineering,Topic_virtual_organizations,Topic_virtual_university,Topic_webbased_Learning,Topic_workflow_management,Topic_www_systems
0,Personen(id1855instance),Forschungsgruppen(id1instance),1,0,0,0,0.00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Personen(id1909instance),Forschungsgruppen(id1instance),0,0,0,0,0.00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Personen(id2040instance),Forschungsgruppen(id3instance),0,6,0,6,2005.00,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Personen(id46instance),Forschungsgruppen(id2instance),1,6,2,1,2004.83,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Personen(id3instance),Forschungsgruppen(id2instance),1,9,2,2,2002.44,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [98]:
# Check Missing Values
feature_df.isnull().sum().sort_values(ascending=False).head(10)

Topic_Computer_Science              67
Topic_Business_models               67
Topic_Computational_Biology         67
Topic_Artificial_Intelligence       67
Topic_AspectOriented_Programming    67
Topic_optimization                  67
Topic_parallel_algorithms           67
Topic_hardware_algorithms           67
Topic_human_computer_systems        67
Topic_hypermedia_systems            67
dtype: int64

In [99]:
# filling null columns with 0
topic_columns = topic_features.columns.drop("Person")

feature_df[topic_columns] = (

    feature_df[topic_columns]

    .fillna(0)

    .astype(int)

)

In [100]:
feature_df.isnull().sum()

Person                         0
Research Group                 0
Homepage                       0
Publication Count              0
WorksAtProject Count           0
                              ..
Topic_virtual_organizations    0
Topic_virtual_university       0
Topic_webbased_Learning        0
Topic_workflow_management      0
Topic_www_systems              0
Length: 133, dtype: int64

### 6.5) Add Conference Features

In [101]:
# ============================================================================
# Collect all unique conferences
# ============================================================================

all_conferences = set()

for conferences in person_conference_df["Conferences"]:

    for conf in conferences:
        all_conferences.add(conf)

all_conferences = sorted(all_conferences)

print(f"Total unique conferences: {len(all_conferences)}")

print("\nFirst 10 conferences:\n")

for conf in all_conferences[:10]:
    print(conf)

Total unique conferences: 686

First 10 conferences:

11.IuK-Jahrestagung 2005
13th European Conference on Information Systems
1999 IEEE Knowledge and Data Engineering Exchange Workshop (KDEX'99), Chicago
19th Workshop on (Constraint) Logic Programming, Ulm, Germany, February 2005
1st Australian Workshop on Engineering Service-Oriented Systems (AWESOS 2004) Wednesday, 14 April 2004, Melbourne, Australia. In conjunction with the Australian Software Engineering Conference (ASWEC)
1st International Workshop on Scripting for the Semantic Web SFSW 2005
1st International Workshop on Semantic Technologies in Collaborative Applications STICA 06
1st Workshop on  Formal Ontologies Meet Industry, FOMI'05, Verona, Italy, June 2005
2007 IEEE Symposium on Foundations of Computational Intelligence
23rd International Conference on Conceptual Modeling (ER2004)


#### 6.5.1 — Conference Frequency Analysis

In [102]:
conference_counter = Counter()

for conferences in person_conference_df["Conferences"]:

    conference_counter.update(conferences)

conference_frequency_df = (
    pd.DataFrame(
        conference_counter.items(),
        columns=["Conference", "Number of Persons"]
    )
    .sort_values("Number of Persons", ascending=False)
    .reset_index(drop=True)
)

conference_frequency_df.head(30)

,Conference,Number of Persons
0,"E-Commerce and Web Technologies, Third Interna...",14
1,Tagungsband des 13. Workshops Algorithmen und ...,9
2,AAAI 2000/IAAI 2000 - Proceedings of the 17th ...,8
3,WWW9 - Proceedings of the 9th International Wo...,8
4,Semantic Web and Peer-to-Peer,8
5,Proceedings of the ISWC 2006 Poster and Demo S...,7
6,WWW8 - Poster Proceedings of the 8th World Wid...,7
7,Proceedings of the W3C Workshop on Frameworks ...,7
8,"INFORMATIK 2005 - Informatik LIVE! Band 2, Bei...",7
9,Proceedings of the ESWC2006 poster and demo se...,7


In [103]:
thresholds = [1, 2, 3, 5, 10]

threshold_summary = []

for t in thresholds:

    count = (
        conference_frequency_df["Number of Persons"] >= t
    ).sum()

    threshold_summary.append({
        "Minimum Persons": t,
        "Remaining Conferences": count
    })

threshold_df = pd.DataFrame(threshold_summary)

threshold_df

,Minimum Persons,Remaining Conferences
0,1,686
1,2,449
2,3,255
3,5,43
4,10,1


In [104]:
# ============================================================================
# Keep only conferences that appear at least 3 times
# ============================================================================

MIN_CONFERENCE_FREQUENCY = 3

frequent_conferences = conference_frequency_df[
    conference_frequency_df["Number of Persons"] >= MIN_CONFERENCE_FREQUENCY
]["Conference"].tolist()

print(f"Number of frequent conferences: {len(frequent_conferences)}")

Number of frequent conferences: 255


In [105]:
# checking conferences actually discriminative for research groups
# ============================================================================
# Conference vs Research Group
# ============================================================================

conference_group_matrix = pd.DataFrame(
    0,
    index=frequent_conferences,
    columns=sorted(person_conference_df["Research Group"].unique())
)

for _, row in person_conference_df.iterrows():

    group = row["Research Group"]

    for conf in row["Conferences"]:

        if conf in conference_group_matrix.index:

            conference_group_matrix.loc[conf, group] += 1

conference_group_matrix.head(20)

,Forschungsgruppen(id1instance),Forschungsgruppen(id2instance),Forschungsgruppen(id3instance),Forschungsgruppen(id4instance)
"E-Commerce and Web Technologies, Third International Conference, EC-Web 2002, Aix-en-Provence, France, September 2-6, 2002, Proceedings",0,0,14,0
Tagungsband des 13. Workshops Algorithmen und Werkzeuge fÃ¼r Petri-Netze (AWPN'06),9,0,0,0
"AAAI 2000/IAAI 2000 - Proceedings of the 17th National Conference on Artificial Intelligence and 12th Innovative Applications of Artificial Intelligence Conference, Austin/TX, USA, July 30-August 3, 2000",0,0,8,0
"WWW9 - Proceedings of the 9th International World Wide Web Conference, Amsterdam, The Netherlands, May, 15-19, 2000",0,0,8,0
Semantic Web and Peer-to-Peer,0,0,8,0
Proceedings of the ISWC 2006 Poster and Demo Session,0,0,7,0
"WWW8 - Poster Proceedings of the 8th World Wide Web Conference, Toronto, May 11-14, 1999",0,0,7,0
Proceedings of the W3C Workshop on Frameworks for Semantics in Web Services,0,0,7,0
"INFORMATIK 2005 - Informatik LIVE! Band 2, BeitrÃ¤ge der 35. Jahrestagung der Gesellschaft fÃ¼r Informatik e.V. (GI), Bonn",3,0,4,0
Proceedings of the ESWC2006 poster and demo session,0,0,7,0


In [106]:
# ============================================================================
# Conference Dominance
# ============================================================================

conference_analysis = conference_group_matrix.copy()

conference_analysis["Maximum Persons"] = conference_analysis.max(axis=1)

conference_analysis["Dominant Group"] = conference_analysis.idxmax(axis=1)

conference_analysis = conference_analysis.sort_values(
    "Maximum Persons",
    ascending=False
)

conference_analysis.head(30)

,Forschungsgruppen(id1instance),Forschungsgruppen(id2instance),Forschungsgruppen(id3instance),Forschungsgruppen(id4instance),Maximum Persons,Dominant Group
"E-Commerce and Web Technologies, Third International Conference, EC-Web 2002, Aix-en-Provence, France, September 2-6, 2002, Proceedings",0,0,14,0,14,Forschungsgruppen(id3instance)
Tagungsband des 13. Workshops Algorithmen und Werkzeuge fÃ¼r Petri-Netze (AWPN'06),9,0,0,0,9,Forschungsgruppen(id1instance)
"AAAI 2000/IAAI 2000 - Proceedings of the 17th National Conference on Artificial Intelligence and 12th Innovative Applications of Artificial Intelligence Conference, Austin/TX, USA, July 30-August 3, 2000",0,0,8,0,8,Forschungsgruppen(id3instance)
"WWW9 - Proceedings of the 9th International World Wide Web Conference, Amsterdam, The Netherlands, May, 15-19, 2000",0,0,8,0,8,Forschungsgruppen(id3instance)
Semantic Web and Peer-to-Peer,0,0,8,0,8,Forschungsgruppen(id3instance)
Proceedings of the ISWC 2006 Poster and Demo Session,0,0,7,0,7,Forschungsgruppen(id3instance)
"WWW8 - Poster Proceedings of the 8th World Wide Web Conference, Toronto, May 11-14, 1999",0,0,7,0,7,Forschungsgruppen(id3instance)
Proceedings of the W3C Workshop on Frameworks for Semantics in Web Services,0,0,7,0,7,Forschungsgruppen(id3instance)
Proceedings of the ESWC2006 poster and demo session,0,0,7,0,7,Forschungsgruppen(id3instance)
"Proceedings of the World Conference on the WWW and Internet (WebNet 99), Honolulu, Hawai, USA, October 25-30, 1999",0,0,7,0,7,Forschungsgruppen(id3instance)


In [107]:
# ============================================================================
# Percentage of conferences dominated by one research group
# ============================================================================

dominance = []

group_columns = conference_group_matrix.columns

for conf in conference_group_matrix.index:

    counts = conference_group_matrix.loc[conf, group_columns]

    total = counts.sum()

    dominant_fraction = counts.max() / total

    dominance.append(dominant_fraction)

conference_analysis["Dominance"] = dominance

conference_analysis.head(20)

,Forschungsgruppen(id1instance),Forschungsgruppen(id2instance),Forschungsgruppen(id3instance),Forschungsgruppen(id4instance),Maximum Persons,Dominant Group,Dominance
"E-Commerce and Web Technologies, Third International Conference, EC-Web 2002, Aix-en-Provence, France, September 2-6, 2002, Proceedings",0,0,14,0,14,Forschungsgruppen(id3instance),1.000000
Tagungsband des 13. Workshops Algorithmen und Werkzeuge fÃ¼r Petri-Netze (AWPN'06),9,0,0,0,9,Forschungsgruppen(id1instance),1.000000
"AAAI 2000/IAAI 2000 - Proceedings of the 17th National Conference on Artificial Intelligence and 12th Innovative Applications of Artificial Intelligence Conference, Austin/TX, USA, July 30-August 3, 2000",0,0,8,0,8,Forschungsgruppen(id3instance),1.000000
"WWW9 - Proceedings of the 9th International World Wide Web Conference, Amsterdam, The Netherlands, May, 15-19, 2000",0,0,8,0,8,Forschungsgruppen(id3instance),1.000000
Semantic Web and Peer-to-Peer,0,0,8,0,8,Forschungsgruppen(id3instance),1.000000
Proceedings of the ISWC 2006 Poster and Demo Session,0,0,7,0,7,Forschungsgruppen(id3instance),1.000000
"WWW8 - Poster Proceedings of the 8th World Wide Web Conference, Toronto, May 11-14, 1999",0,0,7,0,7,Forschungsgruppen(id3instance),1.000000
Proceedings of the W3C Workshop on Frameworks for Semantics in Web Services,0,0,7,0,7,Forschungsgruppen(id3instance),1.000000
Proceedings of the ESWC2006 poster and demo session,0,0,7,0,7,Forschungsgruppen(id3instance),0.571429
"Proceedings of the World Conference on the WWW and Internet (WebNet 99), Honolulu, Hawai, USA, October 25-30, 1999",0,0,7,0,7,Forschungsgruppen(id3instance),1.000000


In [108]:
conference_analysis["Dominance"].describe()

count    255.000000
mean       0.991914
std        0.053356
min        0.571429
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: Dominance, dtype: float64

In [109]:
thresholds = [0.60, 0.70, 0.80, 0.90]

summary = []

for t in thresholds:

    count = (conference_analysis["Dominance"] >= t).sum()

    summary.append({
        "Dominance Threshold": t,
        "Number of Conferences": count
    })

dominance_summary = pd.DataFrame(summary)

dominance_summary

,Dominance Threshold,Number of Conferences
0,0.6,254
1,0.7,251
2,0.8,249
3,0.9,249


**Conclusion : Keep the Top 50 most frequent conferences**

#### Adding only top 50 Conference Features

In [110]:
# ============================================================================
# Select Top 50 Most Frequent Conferences
# ============================================================================

TOP_K = 50

top_conferences = (
    conference_frequency_df
    .head(TOP_K)
    .copy()
)

top_conferences

,Conference,Number of Persons
0,"E-Commerce and Web Technologies, Third Interna...",14
1,Tagungsband des 13. Workshops Algorithmen und ...,9
2,AAAI 2000/IAAI 2000 - Proceedings of the 17th ...,8
3,WWW9 - Proceedings of the 9th International Wo...,8
4,Semantic Web and Peer-to-Peer,8
5,Proceedings of the ISWC 2006 Poster and Demo S...,7
6,WWW8 - Poster Proceedings of the 8th World Wid...,7
7,Proceedings of the W3C Workshop on Frameworks ...,7
8,"INFORMATIK 2005 - Informatik LIVE! Band 2, Bei...",7
9,Proceedings of the ESWC2006 poster and demo se...,7


In [111]:
# ============================================================================
# Conference Feature Mapping
# ============================================================================

conference_lookup = {}

for i, conference in enumerate(top_conferences["Conference"], start=1):

    conference_lookup[conference] = f"Conference_{i:03d}"

conference_mapping_df = pd.DataFrame({

    "Feature Name": conference_lookup.values(),

    "Original Conference": conference_lookup.keys()

})

conference_mapping_df.head(10)

,Feature Name,Original Conference
0,Conference_001,"E-Commerce and Web Technologies, Third Interna..."
1,Conference_002,Tagungsband des 13. Workshops Algorithmen und ...
2,Conference_003,AAAI 2000/IAAI 2000 - Proceedings of the 17th ...
3,Conference_004,WWW9 - Proceedings of the 9th International Wo...
4,Conference_005,Semantic Web and Peer-to-Peer
5,Conference_006,Proceedings of the ISWC 2006 Poster and Demo S...
6,Conference_007,WWW8 - Poster Proceedings of the 8th World Wid...
7,Conference_008,Proceedings of the W3C Workshop on Frameworks ...
8,Conference_009,"INFORMATIK 2005 - Informatik LIVE! Band 2, Bei..."
9,Conference_010,Proceedings of the ESWC2006 poster and demo se...


In [112]:
# ============================================================================
# Multi-hot Encode Conferences
# ============================================================================

conference_matrix = {}

for conference, feature_name in conference_lookup.items():

    conference_matrix[feature_name] = (

        person_conference_df["Conferences"]

        .apply(lambda conferences: int(conference in conferences))

    )

conference_features = pd.concat(

    [

        person_conference_df[["Person"]],

        pd.DataFrame(conference_matrix)

    ],

    axis=1

)

In [113]:
print(conference_features.shape)

conference_features.head()

(116, 51)


,Person,Conference_001,Conference_002,Conference_003,Conference_004,Conference_005,Conference_006,Conference_007,Conference_008,Conference_009,...,Conference_041,Conference_042,Conference_043,Conference_044,Conference_045,Conference_046,Conference_047,Conference_048,Conference_049,Conference_050
0,Personen(id2058instance),0,1,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
1,Personen(id99instance),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Personen(id48instance),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Personen(id2064instance),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Personen(id2062instance),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [114]:
# ============================================================================
# Merge Conference Features
# ============================================================================

feature_df = feature_df.merge(

    conference_features,

    on="Person",

    how="left"

)

In [115]:
# Fill Missing Values
conference_columns = conference_features.columns.drop("Person")

feature_df[conference_columns] = (

    feature_df[conference_columns]

    .fillna(0)

    .astype(int)

)

In [116]:
# Organize Columns
topic_columns = sorted(

    col for col in feature_df.columns

    if col.startswith("Topic_")

)

conference_columns = sorted(

    col for col in feature_df.columns

    if col.startswith("Conference_")

)

ordered_columns = [

    "Person",

    "Homepage",

    "Publication Count",

    "WorksAtProject Count",

    "Number of Projects",

] + topic_columns + conference_columns + [

    "Research Group"

]

feature_df = feature_df[ordered_columns]

In [117]:
#Final Verification
print("="*80)

print("Final Feature Matrix")

print("="*80)

print(f"Rows    : {feature_df.shape[0]}")

print(f"Columns : {feature_df.shape[1]}")

print()

feature_df.head()

Final Feature Matrix
Rows    : 176
Columns : 182



,Person,Homepage,Publication Count,WorksAtProject Count,Number of Projects,Topic_Artificial_Intelligence,Topic_AspectOriented_Programming,Topic_Business_models,Topic_Computational_Biology,Topic_Computer_Science,...,Conference_042,Conference_043,Conference_044,Conference_045,Conference_046,Conference_047,Conference_048,Conference_049,Conference_050,Research Group
0,Personen(id1855instance),1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Forschungsgruppen(id1instance)
1,Personen(id1909instance),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Forschungsgruppen(id1instance)
2,Personen(id2040instance),0,6,0,6,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Forschungsgruppen(id3instance)
3,Personen(id46instance),1,6,2,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Forschungsgruppen(id2instance)
4,Personen(id3instance),1,9,2,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Forschungsgruppen(id2instance)


In [118]:
feature_summary = pd.DataFrame({

    "Feature Group": [

        "Simple Numeric Features",

        "Research Topic Features",

        "Conference Features"

    ],

    "Number of Features": [

        5,      # Homepage + Publication Count + Publication Year +
                # WorksAtProject Count + Number of Projects

        len(topic_columns),

        len(conference_columns)

    ]

})

feature_summary.loc[len(feature_summary)] = [

    "Total Features",

    5 + len(topic_columns) + len(conference_columns)

]

feature_summary

,Feature Group,Number of Features
0,Simple Numeric Features,5
1,Research Topic Features,126
2,Conference Features,50
3,Total Features,181


# 6) Final Feature Matrix

In [119]:
feature_df.head()

,Person,Homepage,Publication Count,WorksAtProject Count,Number of Projects,Topic_Artificial_Intelligence,Topic_AspectOriented_Programming,Topic_Business_models,Topic_Computational_Biology,Topic_Computer_Science,...,Conference_042,Conference_043,Conference_044,Conference_045,Conference_046,Conference_047,Conference_048,Conference_049,Conference_050,Research Group
0,Personen(id1855instance),1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Forschungsgruppen(id1instance)
1,Personen(id1909instance),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Forschungsgruppen(id1instance)
2,Personen(id2040instance),0,6,0,6,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Forschungsgruppen(id3instance)
3,Personen(id46instance),1,6,2,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Forschungsgruppen(id2instance)
4,Personen(id3instance),1,9,2,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Forschungsgruppen(id2instance)


In [120]:
print(f"Rows    : {feature_df.shape[0]}")
print(f"Columns : {feature_df.shape[1]}")

Rows    : 176
Columns : 182


In [121]:
print("Total Missing Values:", feature_df.isnull().sum().sum())

Total Missing Values: 0


In [122]:
print("Duplicate Persons:", feature_df["Person"].duplicated().sum())

Duplicate Persons: 0


In [123]:
# =============================================================================
# Save Final Feature Matrix
# =============================================================================

feature_df.to_csv(
    OUTPUTS_DIR / "feature_matrix.csv",
    index=False
)

feature_df.to_pickle(
    OUTPUTS_DIR / "feature_matrix.pkl"
)

In [124]:
feature_df.to_excel(
    OUTPUTS_DIR / "feature_matrix.xlsx",
    index=False
)

In [125]:
feature_summary.to_csv(
    TABLES_DIR / "feature_summary.csv",
    index=False
)